# Data Methodology

This document explains exactly how every column in every file was calculated, so we can
reason about which fields are safe model features, which are derived/redundant, and which
are generator-internal ground truth that should never be trained on.

Five files come out of the pipeline, generated in this order:

1. `district_indices.csv` - shared market/climate conditions
2. `enterprises.csv` - enterprise master data (safe features)
3. `enterprise_ground_truth.csv` - hidden labels/drivers (evaluation only, **not features**)
4. `monthly_records.csv` - the core monthly panel
5. `transactions.csv` - transaction-level ledger underneath the UPI columns in (4)

Two scripts produce them: `generate_data.py` (files 1–4) runs first, then
`generate_transactions.py` (file 5) reads files 2 and 4, expands the UPI aggregates into
individual transactions, and writes the `upi_outflow_*` and `loan_repayment_collected_upi`
columns back into `monthly_records.csv`.

---

## 1. `district_indices.csv`

One row per (district, month) - 28 districts × 24 months. These are shared conditions that
every enterprise in a district experiences; sector-specific formulas in `monthly_records.csv`
pick out whichever of these indices is relevant to that sector.

All indices are built from a **state-wide baseline curve** (shared across all districts) plus
a **per-district multiplier and noise**, so districts move together but aren't identical.

| Column | How it's calculated |
|---|---|
| `rainfall_index` | `base + amp * sin((month_of_year - 3)/12 * 2π) + noise`, where `base`/`amp`/`noise` depend on the district's `climate_zone` (`arid` < `low` < `medium` < `high`, from `config.py`'s `CLIMATE_ZONE_PARAMS`). This traces a monsoon curve peaking mid-year. 40% of districts additionally get a 3-month "drought shock" (rainfall −20 to −40) starting randomly between month 10–17. Floored at 5.0. |
| `milk_price_index` | A shared state-level sine curve around 100 (±5), with a −12 dip in months 9-11 (flush-season price drop), then a district multiplier (0.95-1.05×) and small independent noise. |
| `poultry_feed_price_index` | A shared state-level random walk around 100 (cumulative sum of small monthly shocks ×0.3) with a +18 spike in months 6-8 (monsoon feed-cost spike), then district multiplier + noise. |
| `raw_material_price_index` | Shared slow upward drift (`100 + 0.4 × month_number`) plus noise, then district multiplier + noise. |
| `retail_demand_index` | Shared baseline of 100, +15 in months 9-10 (festival season demand bump), plus noise, then district multiplier + noise. |
| `local_disruption` | Boolean. ~30% of districts are randomly flagged as ever having a "local disruption." If flagged, one random 1-2 month window gets `retail_demand_index -= 15 to 30` and `raw_material_price_index += 8 to 15` (e.g. a mandi disruption or local price spike), and `local_disruption = True` only in that window. |

All five numeric indices are floored at 5.0 to avoid unrealistic near-zero or negative values.


---

## 2. `enterprises.csv`

One row per enterprise (398 rows) - static/master data. This file is deliberately **safe to
use as model features**: it excludes anything that would leak the stress label (see file 3).

| Column | How it's calculated |
|---|---|
| `enterprise_id` | `"GJ" + first 2 letters of place (uppercased) + 6 random digits`, unique. |
| `name` | Randomly built from one of 3 patterns (proprietor name + business suffix, deity-prefixed name, or cooperative-style name), using name pools in `config.py`. Cosmetic only, not a real signal. |
| `sector` | One of 12 sectors. Sector counts are drawn from a Dirichlet distribution (so sector shares vary dataset-to-dataset) with a floor of 10 enterprises/sector, total enterprise count randomized between 300–400. |
| `district`, `place` | Chosen via `sector_district_choice()`: for sectors with a defined regional cluster (dairy → Anand/Kheda, handicrafts → Kutch, brick_kiln → Vadodara/Bharuch, textile_weaving → Surat, agri_input_retail → several agri districts, poultry → Panchmahal-area), a weighted draw favors those districts; other sectors draw uniformly across all 50 places. |
| `field_investigator_id` | Enterprises are sorted by `(district, place, enterprise_id)` and chunked sequentially into caseloads of ~20 (`FI001`, `FI002`, ...), so each FI's portfolio is geographically contiguous. Any undersized trailing bucket (<10) is folded into the previous FI. This is a fix on top of the original generator - it did not exist before and previously had to be approximated from the enterprise_id prefix. |
| `vintage_years` | `Gamma(shape=2.0, scale=2.5)`, capped at 25 years. Right-skewed: most enterprises are young-to-mid, a smaller tail is long-established. |
| `digital_adoption` | `Normal(mean, 0.18)` clipped to [0.1, 0.95]. Mean is 0.55 for B2B-leaning sectors (brick_kiln, agri_input_retail, auto_repair, flour_mill, oil_mill) and 0.4 for the rest — B2B enterprises adopt UPI somewhat faster/more. |
| `upi_adoption_start_month` | `Normal(6 - digital_adoption×8 + vintage×0.3, sd=3)`, clipped to [0, 22]. Higher digital_adoption and higher vintage both pull this earlier (adopted sooner). This is the month index (0 = first month of data) from which UPI activity starts ramping up. |
| `loan_taken_in_dataset` | Whether the enterprise **ever** has a loan schedule anywhere in the 24 months (fixed master-data fact). `loan_approval_prob = clip(0.35 + min(vintage,10)×0.015 + (performance_multiplier − 1)×0.15, 0.15, 0.9)`, then `has_loan = random() < loan_approval_prob`. Older and better-performing enterprises are more likely to have ever taken a loan. **Not the same as** `has_loan` in `monthly_records.csv`, which is the *per-month active* status (a loan can be taken out or paid off partway through the 24 months). |
| `loan_amount` | Only set if `loan_taken_in_dataset` is true: `Uniform(40000, 180000) × (1 + min(vintage,10)×0.03) × sqrt(max(performance_multiplier, 0.4))`. Bigger, older, better-performing enterprises get bigger loans (sqrt-dampened so it's not a linear blow-up). |
| `loan_tenure_months` | Drawn from `{6, 9, 12, 18, 24, 30, 36, 48}` with probabilities `{0.08, 0.10, 0.20, 0.17, 0.20, 0.10, 0.09, 0.06}` - weighted toward the common 12-24 month range. |
| `loan_start_month_offset` | `Uniform(-(tenure-3), 22)` as an integer - can be negative (loan already underway when the data window opens) up to near the end of the window (just taken out). This produces the mix of loans spanning the full window, loans that close mid-window, and loans that start mid-window. |

---

## 3. `enterprise_ground_truth.csv`

One row per enterprise. **Do not use as model input features** - these are the hidden
variables the generator used to drive stress and performance. Use them only to build/validate
your own stress labels or to sanity-check model output against the true generative process.

| Column | How it's calculated |
|---|---|
| `scripted_stress` | Boolean. Within each sector, exactly the top ~20% (by count, `max(1, int(n×0.2))`) of enterprises are randomly flagged. Flagged enterprises get a scripted income shock in months 10–15: multipliers `[0.95, 0.85, 0.70, 0.65, 0.75, 0.88]` applied to income (i.e. a stress arc that deepens then recovers, peaking around month 13). This is essentially the ground-truth label for "this enterprise goes through a stress episode." |
| `performance_multiplier` | `Lognormal(mean=0, sigma=0.5)` clipped to [0.35, 3.2]. A persistent, enterprise-level "how good is this business, independent of any given month's luck" factor. Multiplies `base_income` every month, and independently influences `loan_approval_prob` and `loan_amount` (better-performing enterprises get approved more often and for more) - this is what creates a real, non-coincidental link between business quality and loan burden. |

---

## 4. `monthly_records.csv`

The core panel: one row per (enterprise, month), 398 × 24 = 9,552 rows. This is where sector,
district conditions, loan schedule, and stress arc all combine into the actual observed
numbers.

**Setup done once per enterprise (before looping over months):**

- `base_income = SECTOR_BASE_INCOME[sector] × performance_multiplier`, then `× (1 + min(vintage,15) × 0.015)` (up to +22.5% for a well-established business).
- `emi`: if the enterprise has an active loan schedule, computed via the standard reducing-balance annuity formula:
  `EMI = P × r × (1+r)^n / ((1+r)^n − 1)`, where `P = loan_amount`, `n = loan_tenure_months`, and `r` = a randomly drawn monthly rate (`annual_rate ~ Uniform(18%, 28%)`, `r = annual_rate/12`). EMI genuinely depends on term length (short loans → high installments, long loans → low installments, both fully amortizing).
- `savings_balance` starts at `Uniform(5000, 25000)`.
- `drift ~ Normal(0, 0.003)`: a tiny enterprise-specific income trend applied over the 24 months.
- `expense_ratio ~ Uniform(0.55, 0.75)`: this enterprise's typical expenses-as-a-fraction-of-income.
- `base_save_rate ~ Uniform(0.08, 0.22)`, `withdrawal_shock_prob ~ Uniform(0.03, 0.09)`, `windfall_prob ~ Uniform(0.02, 0.05)`: persistent per-enterprise savings behavior (see `savings_balance` below).

**Calculated every month:**

| Column | How it's calculated |
|---|---|
| `income` | `base_income × season_mult × shock_mult × scripted_hit × idio_noise × (1 + drift × month_index)`, floored at 15% of `base_income`. See below for each multiplier. |
| - `season_mult` | `SECTOR_SEASONALITY[sector][calendar_month]` - a fixed 12-value seasonal curve per sector from `config.py` (e.g. brick kilns crash in monsoon months, agri input retail spikes at sowing seasons). |
| - `shock_mult` | Sector-specific function of that district's market indices for the month: dairy ~ milk price × rainfall^0.3; poultry ~ (100/feed price)^1.2; food_processing/flour_mill/oil_mill/textile_weaving ~ (100/raw material price)^0.5; handicrafts/rural_retail/tea_stall_eatery ~ retail demand; brick_kiln ~ rainfall^−0.4 (rain hurts kilns); agri_input_retail ~ rainfall^0.3; everything else ~ retail demand^0.5. If `local_disruption` is true that month, multiplied further by `Uniform(0.75, 0.9)`. |
| - `scripted_hit` | 1.0 normally; for `scripted_stress` enterprises in months 10–15, `1 − [0.05, 0.15, 0.30, 0.35, 0.25, 0.12][month−10]` (the stress arc). |
| - `idio_noise` | `Normal(1.0, 0.06)` - plain monthly noise. |
| `expenses` | `income × expense_ratio × Uniform(0.92, 1.08)`. |
| `net_cash_flow` | `income − expenses − emi_due` (exactly; `emi_due` is 0 if no loan is active that month). |
| `savings_balance` | Previous balance + `net_cash_flow × base_save_rate × Uniform(0.6, 1.4)`, plus an occasional shock: with probability `withdrawal_shock_prob`, an emergency draw-down of `−Uniform(15%, 50%)` of the current balance; else with probability `windfall_prob`, an external windfall of `Uniform(2000, 15000)` (money not visible in income/expenses - a gift, family support, small asset sale). Floored at 0. This replaced an earlier version where `savings_balance` was a fixed, noise-free 15%-of-`net_cash_flow` accumulation (making it a redundant rescaling of the target); it now behaves like a semi-independent quantity. |
| `has_loan` | Whether a loan schedule is **active this specific month**: `loan_start_offset ≤ month_index < loan_start_offset + loan_tenure_months`. (Different from `loan_taken_in_dataset` in `enterprises.csv`, which just means "ever has a schedule.") |
| `emi_due` | The EMI value computed above, if `has_loan` this month; else 0. |
| `repayment_status` | `no_loan` if the enterprise never has a loan, or the loan hasn't started yet. `loan_closed` if the loan's tenure has completed. Otherwise (`loan_active`), computed from `covered = (income − expenses) + min(savings_balance, emi_due) × 0.3` - how much of the EMI the enterprise can plausibly cover from cash flow plus a partial savings buffer: `missed` if `covered < 0.7 × emi_due`, `late` if `covered < 1.15 × emi_due`, else `on_time`. |
| `upi_inflow_txn_count`, `upi_inflow_txn_volume` | `effective_adoption = digital_adoption × ramp`, where `ramp = min(1, (month_index − upi_adoption_start_month) / 4)` (0 before adoption starts, ramping up over 4 months, 0.02 as a floor pre-adoption). `upi_inflow_txn_volume = income × effective_adoption × Uniform(0.85, 1.15)`. `upi_inflow_txn_count = int((income / 900) × effective_adoption × Uniform(0.8, 1.2))`, with a floor of 1 transaction whenever volume is nonzero (a transaction count of 0 with nonzero volume is a contradiction). |
| `data_complete` | `random() > underreport_prob`, where `underreport_prob = 0.08 × (1 − digital_adoption)`. Low-digital-adoption enterprise-months are more likely to be flagged incomplete - models partial field-data capture without introducing raw NaNs. |
| `field_investigator_id` | Joined in from `enterprises.csv` (same value as that enterprise's row there - it's a fixed master-data attribute, not month-varying). |
| `upi_outflow_txn_count`, `upi_outflow_txn_volume`, `loan_repayment_collected_upi` | **Not generated in this file directly** - these three are computed from `transactions.csv` and merged back in. See file 5 below. |

---

## 5. `transactions.csv`

One row per individual UPI transaction (~160k rows). Generated by `generate_transactions.py`,
which reads `monthly_records.csv` + `enterprises.csv` and expands each enterprise-month's
aggregate counts/volumes into individual timestamped transactions - useful for intra-month
features (day-of-week concentration, burstiness, counterparty mix) that the monthly file
can't express.

| Column | How it's calculated |
|---|---|
| `transaction_id` | Sequential `TXN00000001`, `TXN00000002`, ... after sorting all transactions by `(enterprise_id, timestamp)`. |
| `enterprise_id`, `month` | Carried over from the source `monthly_records.csv` row. |
| `timestamp` | Day: sampled with weights favoring the first half of the month (1.3× for days 1-10, 1.1× for days 11-20, 0.8× for days 21–end) - income/payments cluster around paydays. Hour: sampled from 8am-8pm with a peak around midday and early evening (not uniform - rural micro-enterprises don't transact at 3am). For EMI repayment transactions specifically, the day is biased into the back half of the month, hour restricted to 9am-6pm. |
| `direction` | `inflow` (money received), `outflow` (money paid out digitally). |
| `amount` | **Inflows:** that month's `upi_inflow_txn_count` transactions are drawn from a lognormal shape (`σ=0.7`) then rescaled to sum exactly to `upi_inflow_txn_volume` - so the two files always reconcile exactly by construction. **Outflows (non-EMI):** count is `int(n_inflow × Uniform(0.25, 0.5))` (fewer than inflows), volume is `expenses × Uniform(0.35, 0.65) × (upi_inflow_txn_volume / income)` (a "digital share of expenses" scaled by how digitally active that enterprise already is), then split across transactions the same lognormal way. **EMI repayment:** its own dedicated transaction (see below), not sampled from the general outflow pool. |
| `counterparty_type` | **Inflows:** sampled per-sector from a fixed pool (e.g. dairy → mostly `cooperative_payment`/`customer_payment`, brick_kiln → `wholesale_buyer_payment`/`contractor_payment`), each with fixed relative weights. **Outflows:** sampled from `{supplier_payment, utility_bill, wage_payment, transport_payment}` with weights `{0.45, 0.15, 0.30, 0.10}`. **EMI:** always `loan_repayment`. |

**EMI transaction logic (`_emi_transaction`)** - gated strictly on that month's `repayment_status`, not sampled independently, so a `loan_repayment` transaction can never appear for an enterprise with no active loan that month:

- `repayment_status == "on_time"` → one transaction for the full `emi_due` amount.
- `repayment_status == "late"` → one transaction for `Uniform(40%, 85%)` of `emi_due`, timestamped in the back half of the month.
- `repayment_status == "missed"` → no transaction at all.
- `no_loan` / `loan_closed` → no transaction, always.
- Even when `on_time`/`late`, there's an 8% chance (`EMI_CASH_PAYMENT_PROB`) the payment happened at a branch/agent in cash rather than via UPI - so it's silently skipped from the digital ledger even though `repayment_status` still reflects the true (non-digital) repayment behavior.

**Reconciliation back into `monthly_records.csv`:** after building `transactions.csv`, the
script re-aggregates all `outflow` rows by `(enterprise_id, month)` to produce
`upi_outflow_txn_count` and `upi_outflow_txn_volume`, and separately sums `loan_repayment`
rows to produce `loan_repayment_collected_upi`. These three columns are merged into
`monthly_records.csv` (filling 0 where an enterprise-month had no outflow transactions at
all), so they're guaranteed consistent with the ledger rather than being a second,
independently-generated number.

---

## Quick reference: derived vs. independent columns

Useful to know which columns are exact functions of others (no new information) vs. which
carry genuinely separate signal:

- **Exact identities:** `net_cash_flow = income − expenses − emi_due`. `upi_outflow_txn_volume` = sum of that month's outflow transactions in `transactions.csv`. `loan_repayment_collected_upi` = sum of that month's `loan_repayment` transactions.
- **Noisy but derived:** `savings_balance` (function of `net_cash_flow` history plus its own noise/shocks - correlated with, but not reconstructable from, cash flow alone). `repayment_status` (function of cash flow + savings + emi_due, with thresholds).
- **Ground truth, not features:** `scripted_stress`, `performance_multiplier` - only in `enterprise_ground_truth.csv`, kept separate for exactly this reason.
